# PubMedQA LoRA+ on Kaggle

Self-contained notebook for Kaggle training on two T4 GPUs.

Defaults:
- masked-answer supervision on PubMedQA
- LoRA+ optimizer with separate A/B learning rates
- full-precision loading with a smaller instruct model by default
- adapter and merged-model export


In [ ]:
import subprocess
import sys

subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "-U",
        "accelerate",
        "datasets",
        "huggingface_hub",
        "peft",
        "scikit-learn",
        "sentencepiece",
        "torch",
        "transformers",
        "wandb",
    ]
)


In [ ]:
import json
import os
import random
from pathlib import Path

import numpy as np
import torch
from accelerate import notebook_launcher
from datasets import load_dataset
from huggingface_hub import HfApi, login
from peft import LoraConfig, PeftModel, get_peft_model
from peft.optimizers import create_loraplus_optimizer
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score
from transformers import AutoModelForCausalLM, AutoTokenizer, Trainer, TrainingArguments

try:
    from kaggle_secrets import UserSecretsClient

    secrets = UserSecretsClient()
    for secret_name in ("HF_TOKEN", "WANDB_API_KEY"):
        try:
            os.environ[secret_name] = secrets.get_secret(secret_name)
        except Exception:
            pass
except Exception:
    pass

if os.getenv("HF_TOKEN"):
    login(token=os.environ["HF_TOKEN"], add_to_git_credential=False)

WORKDIR = Path("/kaggle/working/pubmedqa_loraplus")
OUTPUT_DIR = WORKDIR / "outputs"
REPORT_DIR = WORKDIR / "reports"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

CONFIG = {
    "run_name": "pubmedqa-loraplus-kaggle",
    "base_model_id": os.getenv("BASE_MODEL_ID", "Qwen/Qwen2.5-1.5B-Instruct"),
    "dataset_id": "qiaojin/PubMedQA",
    "dataset_config": "pqa_labeled",
    "seed": 42,
    "test_size": 0.1,
    "max_length": 768,
    "train_batch_size": 1,
    "eval_batch_size": 1,
    "gradient_accumulation_steps": 16,
    "learning_rate": 5e-5,
    "loraplus_lr_ratio": 16.0,
    "loraplus_lr_embedding": 1e-6,
    "num_epochs": 3,
    "logging_steps": 10,
    "save_steps": 100,
    "eval_steps": 100,
    "lora_r": 16,
    "lora_alpha": 32,
    "lora_dropout": 0.05,
    "target_modules": ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    "warmup_ratio": 0.03,
    "weight_decay": 0.0,
    "lr_scheduler_type": "cosine",
    "max_grad_norm": 1.0,
    "mmlu_eval_samples": 50,
    "adapter_hub_repo_id": os.getenv("ADAPTER_HUB_REPO_ID"),
    "merged_hub_repo_id": os.getenv("MERGED_HUB_REPO_ID"),
}

DEVICE_COUNT = torch.cuda.device_count()
print({"device_count": DEVICE_COUNT, "base_model": CONFIG["base_model_id"], "workdir": str(WORKDIR)})
if DEVICE_COUNT < 2:
    raise RuntimeError("This notebook is intended to run on Kaggle with two T4 GPUs.")

(WORKDIR / "config.json").write_text(json.dumps(CONFIG, indent=2), encoding="utf-8")


In [ ]:
dataset = load_dataset(CONFIG["dataset_id"], CONFIG["dataset_config"])["train"].train_test_split(
    test_size=CONFIG["test_size"],
    seed=CONFIG["seed"],
)
print(dataset)
print(dataset["train"][0])


In [ ]:
PUBMEDQA_LABELS = ("yes", "no", "maybe")
MMLU_CHOICES = ("A", "B", "C", "D")
MMLU_MEDICAL_TASKS = (
    "anatomy",
    "clinical_knowledge",
    "college_medicine",
    "medical_genetics",
    "professional_medicine",
    "virology",
)

def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def flatten_context(context: object) -> str:
    if isinstance(context, dict) and "contexts" in context:
        return " ".join(context["contexts"])
    if isinstance(context, list):
        return " ".join(str(part) for part in context)
    return str(context)


def build_instruction_prompt(question: str, context: str, bos_token: str) -> str:
    return f"{bos_token}[INST] Context: {context}\n\nQuestion: {question} [/INST]"


def build_answer_text(answer: str, eos_token: str) -> str:
    return f" {answer.lower()}{eos_token}"


def tokenize_example(example, tokenizer):
    prompt = build_instruction_prompt(
        example["question"],
        flatten_context(example["context"]),
        tokenizer.bos_token or "<s>",
    )
    answer = build_answer_text(example["final_decision"], tokenizer.eos_token or "</s>")
    prompt_ids = tokenizer(prompt, add_special_tokens=False)["input_ids"]
    answer_ids = tokenizer(answer, add_special_tokens=False)["input_ids"]
    max_prompt_tokens = max(CONFIG["max_length"] - len(answer_ids), 1)
    prompt_ids = prompt_ids[:max_prompt_tokens]
    input_ids = (prompt_ids + answer_ids)[: CONFIG["max_length"]]
    return {
        "input_ids": input_ids,
        "attention_mask": [1] * len(input_ids),
        "labels": ([-100] * len(prompt_ids) + answer_ids)[: CONFIG["max_length"]],
    }


class CompletionOnlyCollator:
    def __init__(self, pad_token_id: int):
        self.pad_token_id = pad_token_id

    def __call__(self, features):
        max_length = max(len(feature["input_ids"]) for feature in features)
        input_ids, attention_mask, labels = [], [], []
        for feature in features:
            padding = max_length - len(feature["input_ids"])
            input_ids.append(feature["input_ids"] + [self.pad_token_id] * padding)
            attention_mask.append(feature["attention_mask"] + [0] * padding)
            labels.append(feature["labels"] + [-100] * padding)
        return {
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
            "labels": torch.tensor(labels, dtype=torch.long),
        }


def score_candidate(model, tokenizer, prompt: str, candidate: str) -> float:
    prompt_ids = tokenizer(prompt, add_special_tokens=False)["input_ids"]
    candidate_ids = tokenizer(candidate, add_special_tokens=False)["input_ids"]
    max_prompt_tokens = max(CONFIG["max_length"] - len(candidate_ids), 1)
    prompt_ids = prompt_ids[:max_prompt_tokens]
    input_ids = torch.tensor([prompt_ids + candidate_ids], dtype=torch.long, device=next(model.parameters()).device)
    attention_mask = torch.ones_like(input_ids)
    with torch.no_grad():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
    logits = outputs.logits[0]
    log_probs = torch.log_softmax(logits, dim=-1)
    score = 0.0
    prompt_offset = len(prompt_ids)
    for token_index, token_id in enumerate(candidate_ids):
        score += float(log_probs[prompt_offset + token_index, token_id].item())
    return score


def predict_from_candidates(model, tokenizer, prompt: str, candidates: dict[str, str]) -> str:
    scores = {label: score_candidate(model, tokenizer, prompt, candidate) for label, candidate in candidates.items()}
    return max(scores, key=scores.get)


def evaluate_pubmedqa(model, tokenizer, eval_dataset):
    predictions, references = [], []
    label_candidates = {
        label: build_answer_text(label, tokenizer.eos_token or "</s>")
        for label in PUBMEDQA_LABELS
    }
    for example in eval_dataset:
        prompt = build_instruction_prompt(
            example["question"],
            flatten_context(example["context"]),
            tokenizer.bos_token or "<s>",
        )
        predictions.append(predict_from_candidates(model, tokenizer, prompt, label_candidates))
        references.append(example["final_decision"].lower())
    cm = confusion_matrix(references, predictions, labels=list(PUBMEDQA_LABELS))
    return {
        "accuracy": accuracy_score(references, predictions),
        "macro_f1": f1_score(references, predictions, average="macro"),
        "weighted_f1": f1_score(references, predictions, average="weighted"),
        "confusion_matrix": cm.tolist(),
        "num_samples": len(predictions),
    }


def evaluate_mmlu_medical(model, tokenizer):
    mmlu = load_dataset("cais/mmlu", "all", split="test")
    medical = mmlu.filter(lambda row: row["subject"] in MMLU_MEDICAL_TASKS)
    medical = medical.select(range(min(CONFIG["mmlu_eval_samples"], len(medical))))
    subject_results = {subject: {"correct": 0, "total": 0} for subject in MMLU_MEDICAL_TASKS}
    for example in medical:
        prompt = "The following are multiple choice questions about medical knowledge.\n"
        prompt += f"Question: {example['question']}\n"
        for index, choice in enumerate(example["choices"]):
            prompt += f"{chr(65 + index)}. {choice}\n"
        prompt += "Answer:"
        prediction = predict_from_candidates(
            model,
            tokenizer,
            prompt,
            {choice: f" {choice}" for choice in MMLU_CHOICES},
        )
        subject = example["subject"]
        subject_results[subject]["total"] += 1
        if ord(prediction) - ord("A") == example["answer"]:
            subject_results[subject]["correct"] += 1
    total_correct = sum(result["correct"] for result in subject_results.values())
    total_samples = sum(result["total"] for result in subject_results.values())
    return {
        "overall_accuracy": total_correct / total_samples if total_samples else 0.0,
        "subject_results": subject_results,
        "total_samples": total_samples,
    }


In [ ]:
class NotebookLoRAPlusTrainer(Trainer):
    def create_optimizer(self):
        if self.optimizer is None:
            self.optimizer = create_loraplus_optimizer(
                self.model,
                torch.optim.AdamW,
                lr=self.args.learning_rate,
                loraplus_lr_ratio=CONFIG["loraplus_lr_ratio"],
                loraplus_lr_embedding=CONFIG["loraplus_lr_embedding"],
                loraplus_weight_decay=self.args.weight_decay,
                betas=(self.args.adam_beta1, self.args.adam_beta2),
                eps=self.args.adam_epsilon,
            )
        return self.optimizer


def train_worker(config: dict):
    set_seed(config["seed"])
    tokenizer = AutoTokenizer.from_pretrained(config["base_model_id"])
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"

    train_split = load_dataset(config["dataset_id"], config["dataset_config"])["train"].train_test_split(
        test_size=config["test_size"],
        seed=config["seed"],
    )
    tokenized = train_split.map(
        lambda example: tokenize_example(example, tokenizer),
        remove_columns=train_split["train"].column_names,
    )

    model = AutoModelForCausalLM.from_pretrained(
        config["base_model_id"],
        torch_dtype=torch.float16,
        trust_remote_code=True,
    )
    model.config.use_cache = False
    model = get_peft_model(
        model,
        LoraConfig(
            r=config["lora_r"],
            lora_alpha=config["lora_alpha"],
            lora_dropout=config["lora_dropout"],
            bias="none",
            task_type="CAUSAL_LM",
            target_modules=config["target_modules"],
        ),
    )

    trainer = NotebookLoRAPlusTrainer(
        model=model,
        args=TrainingArguments(
            output_dir=str(OUTPUT_DIR / "checkpoints"),
            per_device_train_batch_size=config["train_batch_size"],
            per_device_eval_batch_size=config["eval_batch_size"],
            gradient_accumulation_steps=config["gradient_accumulation_steps"],
            learning_rate=config["learning_rate"],
            num_train_epochs=config["num_epochs"],
            logging_steps=config["logging_steps"],
            save_steps=config["save_steps"],
            eval_strategy="steps",
            eval_steps=config["eval_steps"],
            fp16=True,
            gradient_checkpointing=True,
            ddp_find_unused_parameters=False,
            remove_unused_columns=False,
            label_names=["labels"],
            load_best_model_at_end=True,
            metric_for_best_model="eval_loss",
            greater_is_better=False,
            save_total_limit=2,
            warmup_ratio=config["warmup_ratio"],
            weight_decay=config["weight_decay"],
            lr_scheduler_type=config["lr_scheduler_type"],
            max_grad_norm=config["max_grad_norm"],
            report_to=["wandb"] if os.getenv("WANDB_API_KEY") or os.getenv("WANDB_MODE", "").lower() == "offline" else [],
            run_name=config["run_name"],
        ),
        train_dataset=tokenized["train"],
        eval_dataset=tokenized["test"],
        data_collator=CompletionOnlyCollator(tokenizer.pad_token_id),
    )
    trainer.train()
    trainer.save_model(str(OUTPUT_DIR / "adapter"))
    tokenizer.save_pretrained(str(OUTPUT_DIR / "tokenizer"))


notebook_launcher(train_worker, (CONFIG,), num_processes=2)


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(str(OUTPUT_DIR / "tokenizer"))
base_model = AutoModelForCausalLM.from_pretrained(
    CONFIG["base_model_id"],
    torch_dtype=torch.float16,
    trust_remote_code=True,
).to("cuda")
adapter_model = PeftModel.from_pretrained(base_model, str(OUTPUT_DIR / "adapter")).to("cuda")

results = {
    "pubmedqa": evaluate_pubmedqa(adapter_model, tokenizer, dataset["test"]),
    "mmlu_medical": evaluate_mmlu_medical(adapter_model, tokenizer),
}
report = {
    "config": CONFIG,
    "artifacts": {
        "adapter": str(OUTPUT_DIR / "adapter"),
        "tokenizer": str(OUTPUT_DIR / "tokenizer"),
    },
    "evaluation": results,
}
(REPORT_DIR / "pubmedqa_loraplus_kaggle.json").write_text(json.dumps(report, indent=2), encoding="utf-8")
report


In [ ]:
merged_model = adapter_model.merge_and_unload()
merged_dir = OUTPUT_DIR / "merged"
merged_dir.mkdir(parents=True, exist_ok=True)
merged_model.save_pretrained(str(merged_dir), safe_serialization=True)
tokenizer.save_pretrained(str(merged_dir))
print({"adapter": str(OUTPUT_DIR / 'adapter'), "merged": str(merged_dir)})
